# Day 26 — Dashboard Route and API Integration

This notebook inspects the existing Django URL configuration and adds a separate dashboard view plus a URL patch. It does not overwrite existing `views.py` or `urls.py`. Run from the repository root in the `ipl-player-performance` environment.

Day 25 created `dashboard_day25.html`, `dashboard_day25.css`, and `dashboard_day25.js`. The route and API must still be verified in a browser.

In [29]:
from pathlib import Path
import subprocess, sys, shutil

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots: REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
URLS_PY = APP_DIR / "urls.py"
TEMPLATE = APP_DIR / "templates" / "ipl_api" / "dashboard_day25.html"
assert MANAGE_PY.exists(), f"Missing {MANAGE_PY}"
assert URLS_PY.exists(), f"Missing {URLS_PY}"
assert TEMPLATE.exists(), f"Run Day 25 first; missing {TEMPLATE}"
print("Repository:", REPO_ROOT)
print("Existing urls.py:", URLS_PY)
print("Dashboard template:", TEMPLATE)
print("\n--- Existing urls.py (read-only preview) ---")
print(URLS_PY.read_text(encoding="utf-8"))


Repository: D:\Git\Personal\ipl-player-performance
Existing urls.py: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Dashboard template: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html

--- Existing urls.py (read-only preview) ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda

In [30]:
view_source = 'from django.shortcuts import render\n\ndef dashboard_day25(request):\n    """Render the IPL analytics dashboard shell."""\n    return render(request, "ipl_api/dashboard_day25.html")\n'
compile(view_source, "dashboard_views_day26.py", "exec")
view_path = APP_DIR / "dashboard_views_day26.py"
if view_path.exists():
    shutil.copy2(view_path, view_path.with_suffix(".py.day26_backup"))
view_path.write_text(view_source, encoding="utf-8")
print("Created:", view_path)


Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\dashboard_views_day26.py


## Final root URL repair and automatic documentation
This cell deterministically replaces any existing root `path('', ...)` declaration, registers the root redirect, runs Django checks, resolves the actual URLs using Django's runtime resolver, runs the Day 21 regression tests, and generates `docs/day26_api_dashboard_repair_report.md`.


In [31]:
from pathlib import Path
import re, shutil, datetime, subprocess, sys

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "django_api" / "manage.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the IPL project repository.")
DJANGO_ROOT = REPO_ROOT / "django_api"
PROJECT_URLS = DJANGO_ROOT / "config" / "urls.py"
APP_URLS = DJANGO_ROOT / "ipl_api" / "urls.py"
MANAGE_PY = DJANGO_ROOT / "manage.py"
DOCS = REPO_ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
REPORT = DOCS / "day26_api_dashboard_repair_report.md"
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

if not PROJECT_URLS.is_file() or not APP_URLS.is_file():
    raise FileNotFoundError(f"Expected URL files not found: {PROJECT_URLS} and/or {APP_URLS}")

def backup(path):
    dest = path.with_name(path.name + ".day26_verified_backup_" + stamp)
    shutil.copy2(path, dest)
    print("Backup created:", dest)

before = PROJECT_URLS.read_text(encoding="utf-8")
text = before
if not re.search(r"(?m)^\s*from\s+django\.urls\s+import\s+.*\bpath\b", text):
    raise RuntimeError("config/urls.py does not import path from django.urls. Inspect it before modifying.")

if not re.search(r"(?m)^\s*from\s+django\.views\.generic\.base\s+import\s+.*\bRedirectView\b", text):
    if "from django.urls import path, include" in text:
        text = text.replace("from django.urls import path, include",
                            "from django.urls import path, include\nfrom django.views.generic.base import RedirectView", 1)
    else:
        imports = list(re.finditer(r"(?m)^from django\.urls import .+$", text))
        if imports:
            pos = imports[-1].end()
            text = text[:pos] + "\nfrom django.views.generic.base import RedirectView" + text[pos:]
        else:
            text = "from django.views.generic.base import RedirectView\n" + text

if not re.search(r"(?m)^\s*urlpatterns\s*=\s*\[", text):
    raise RuntimeError("Could not find urlpatterns = [ in config/urls.py; no route was written.")

root_line = '    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home-dashboard"),'
lines = text.splitlines()
clean_lines = []
for line in lines:
    if re.match(r"^\s*path\(\s*(['\"])\s*\1\s*,", line):
        continue
    if re.match(r"^\s*path\(", line) and "home-dashboard" in line:
        continue
    clean_lines.append(line)
text = "\n".join(clean_lines) + ("\n" if before.endswith("\n") else "")
urlpatterns_match = re.search(r"(?m)^\s*urlpatterns\s*=\s*\[", text)
text = text[:urlpatterns_match.end()] + "\n" + root_line + text[urlpatterns_match.end():]
compile(text, str(PROJECT_URLS), "exec")
if text != before:
    backup(PROJECT_URLS)
    PROJECT_URLS.write_text(text, encoding="utf-8")
    print("Updated project root route:", PROJECT_URLS)
else:
    print("Project URL file already has the desired route; rewritten content is identical.")

project_text = PROJECT_URLS.read_text(encoding="utf-8")
app_text = APP_URLS.read_text(encoding="utf-8")
root_pattern = re.compile(r"(?m)^\s*path\(\s*['\"]\s*['\"]\s*,\s*RedirectView\.as_view\(\s*url=['\"]/api/dashboard/['\"]")
root_ok = bool(root_pattern.search(project_text))
dashboard_ok = bool(re.search(r"path\(\s*['\"]dashboard/['\"]", app_text)) and "dashboard_day25" in app_text
summary_ok = bool(re.search(r"path\(\s*['\"]eda/summary/['\"]", app_text)) and "eda_summary_day26" in app_text
print("Root redirect source check:", "PASS" if root_ok else "FAIL")
print("Dashboard app route source check:", "PASS" if dashboard_ok else "FAIL")
print("EDA summary app route source check:", "PASS" if summary_ok else "FAIL")

check = subprocess.run([sys.executable, str(MANAGE_PY), "check"], cwd=str(DJANGO_ROOT), capture_output=True, text=True)
print("\n$ python manage.py check\nExit code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)

resolver_code = (
    "import os; os.environ.setdefault('DJANGO_SETTINGS_MODULE','config.settings'); "
    "import django; django.setup(); "
    "from django.urls import resolve; "
    "print('ROOT_RESOLVES_TO=' + str(resolve('/').url_name)); "
    "print('DASHBOARD_RESOLVES_TO=' + str(resolve('/api/dashboard/').url_name)); "
    "print('SUMMARY_RESOLVES_TO=' + str(resolve('/api/eda/summary/').url_name))"
)
resolve_result = subprocess.run([sys.executable, "-c", resolver_code], cwd=str(DJANGO_ROOT), capture_output=True, text=True)
print("\n$ Django URL resolver verification\nExit code:", resolve_result.returncode)
print(resolve_result.stdout)
if resolve_result.stderr:
    print(resolve_result.stderr)

tests = subprocess.run(
    [sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"],
    cwd=str(DJANGO_ROOT), capture_output=True, text=True
)
print("\n$ Day 21 regression tests\nExit code:", tests.returncode)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)

runtime_ok = (
    resolve_result.returncode == 0
    and "ROOT_RESOLVES_TO=home-dashboard" in resolve_result.stdout
    and "DASHBOARD_RESOLVES_TO=ipl-dashboard" in resolve_result.stdout
    and "SUMMARY_RESOLVES_TO=eda-summary-day26" in resolve_result.stdout
)
report = f"""# Day 26 — Dashboard and API URL Repair

Generated: {datetime.datetime.now().isoformat(timespec="seconds")}

## Verified results
- Root route source check: {"PASS" if root_ok else "FAIL"}
- Dashboard route source check: {"PASS" if dashboard_ok else "FAIL"}
- EDA summary route source check: {"PASS" if summary_ok else "FAIL"}
- Django URL resolver runtime check: {"PASS" if runtime_ok else "FAIL"}
- `python manage.py check`: {"PASS" if check.returncode == 0 else "FAIL"} (exit code {check.returncode})
- Day 21 regression tests: {"PASS" if tests.returncode == 0 else "FAIL"} (exit code {tests.returncode})

## Expected routes
- `/` → redirects to `/api/dashboard/`
- `/api/dashboard/` → dashboard page
- `/api/eda/summary/` → EDA summary JSON

## Runtime limitation
URL resolution does not guarantee that the dashboard renders or that MySQL queries succeed. After this notebook runs, restart the development server and test the browser routes. If the browser still shows only `api/` and `admin/` in the root 404 page, the running server is likely using a different checkout/settings module than the repository this notebook edited; compare the printed absolute paths with the terminal working directory.

## Commands
Run from `django_api`:
`python manage.py runserver`
"""
REPORT.write_text(report, encoding="utf-8")
print("\nDocumentation generated:", REPORT)
print("Report path:", REPORT)
if not (root_ok and dashboard_ok and summary_ok and check.returncode == 0 and runtime_ok and tests.returncode == 0):
    raise RuntimeError("One or more checks failed. Read the outputs and generated Markdown report before proceeding.")


Project URL file already has the desired route; rewritten content is identical.
Root redirect source check: PASS
Dashboard app route source check: PASS
EDA summary app route source check: PASS

$ python manage.py check
Exit code: 0
System check identified no issues (0 silenced).


$ Django URL resolver verification
Exit code: 0
ROOT_RESOLVES_TO=home-dashboard
DASHBOARD_RESOLVES_TO=ipl-dashboard
SUMMARY_RESOLVES_TO=eda-summary-day26


$ Day 21 regression tests
Exit code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK


Documentation generated: D:\Git\Personal\ipl-player-performance\docs\day26_api_dashboard_repair_report.md
Report path: D:\Git\Personal\ipl-player-performance\docs\day26_api_dashboard_repair_report.md


In [32]:
report = f'''# Day 26 — Dashboard Route Integration

## Generated files
- `django_api/ipl_api/dashboard_views_day26.py`
- `django_api/ipl_api/day26_dashboard_url_patch.txt`

## Automated checks
- View syntax: passed.
- `manage.py check` return code: {django_check.returncode}.
- Day 21 regression test return code: {regression.returncode}.

## Manual integration still required
1. Merge the URL import and dashboard path into the existing `urls.py`.
2. Run `python manage.py check`.
3. Run `python manage.py runserver`.
4. Open `http://127.0.0.1:8000/dashboard/`.
5. Click Refresh summary and inspect the browser result for `/api/eda/summary/`.
6. Update metric mappings only after observing the actual JSON response.

Automated checks do not prove the route was merged, the template renders in a browser, or the API call succeeds.
'''
(REPO_ROOT / "docs").mkdir(exist_ok=True)
(REPO_ROOT / "docs" / "day26_dashboard_api_integration.md").write_text(report, encoding="utf-8")
print("Saved docs/day26_dashboard_api_integration.md")


Saved docs/day26_dashboard_api_integration.md


## Manual test checklist

- [ ] Merge `day26_dashboard_url_patch.txt` into existing `urls.py`.
- [ ] Preserve all Day 23/24 and existing routes.
- [ ] Run `python manage.py check`.
- [ ] Start the server and open `/dashboard/`.
- [ ] Verify template and CSS load.
- [ ] Click Refresh summary and record the actual JSON response or error.
- [ ] Upload the notebook output and browser console/network result if the API call fails.

Do not proceed to charts and prediction forms until this route is confirmed working.